<a href="https://colab.research.google.com/github/BoutalebAnas/Lab-2-SparkSQL_and_dataframes/blob/main/lab_sparksql_and_dataframes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
yoda

# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [ ]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [ ]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [ ]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [ ]:
df_trips.createOrReplaceTempView("trips")

# Utiliser Spark SQL
result = spark.sql("""
    SELECT *
    FROM trips
    LIMIT 10
""")

result.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [ ]:
# Add a column that creates a unique key to identify each record in order to answer questions about individual trips
from pyspark.sql.functions import monotonically_increasing_id
df_trips = df_trips.withColumn(
    "trip_id",
    monotonically_increasing_id()
)

df_trips.createOrReplaceTempView("trips")

spark.sql("""
    SELECT trip_id, *
    FROM trips
    LIMIT 10
""").show()

+-------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|trip_id|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|
+-------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|      0|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|     

In [70]:
from pyspark.sql.functions import min, max, col, to_date, count, hour, dayofweek, avg
# Which trip has the highest passanger count
df_trips.orderBy(
    df_trips.passenger_count.desc()
).select(
    "trip_id",
    "passenger_count",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
).limit(1).show()

+-------+---------------+--------------------+---------------------+------------+------------+
|trip_id|passenger_count|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|
+-------+---------------+--------------------+---------------------+------------+------------+
| 949956|            9.0| 2019-01-05 13:12:29|  2019-01-05 13:12:32|          68|          68|
+-------+---------------+--------------------+---------------------+------------+------------+



In [ ]:
# What is the Average passanger count
spark.sql("""
    SELECT AVG(passenger_count) AS average_passenger_count
    FROM trips
""").show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670317144945614|
+-----------------------+



In [ ]:
# Shortest/longest trip by distance? by time?.
df_trips.agg(
    min("trip_distance").alias("shortest_trip"),
    max("trip_distance").alias("longest_trip")
).show()

+-------------+------------+
|shortest_trip|longest_trip|
+-------------+------------+
|          0.0|       831.8|
+-------------+------------+



In [ ]:
# Busiest day/slowest single day
df_daily = df_trips.withColumn(
    "day", to_date("tpep_pickup_datetime")
    ).groupBy(
        "day"
    ). agg(count("*").alias("num_trips"))
# Busiest Day
df_daily.orderBy(
    col("num_trips").asc()
).limit(1).show()
# Slowest Day
df_daily.orderBy(
    col("num_trips").desc()
).limit(1).show()

+----------+---------+
|       day|num_trips|
+----------+---------+
|2019-02-23|        1|
+----------+---------+

+----------+---------+
|       day|num_trips|
+----------+---------+
|2019-01-25|   292499|
+----------+---------+



In [ ]:
# Busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
df_hour = df_trips.withColumn(
    "hour", hour("tpep_pickup_datetime")
).groupBy(
    "hour"
). agg(count("*").alias("num_trips"))

# Busiest Day
df_hour.orderBy(col("num_trips").desc()).limit(1).show()

# Slowest Day
df_hour.orderBy(col("num_trips").asc()).limit(1).show()

+----+---------+
|hour|num_trips|
+----+---------+
|  18|   515390|
+----+---------+

+----+---------+
|hour|num_trips|
+----+---------+
|   4|    61424|
+----+---------+



In [ ]:
# On average which day of the week is slowest/busiest
df_weekday = df_trips.withColumn(
    "day_of_week",
    dayofweek("tpep_pickup_datetime")
).groupBy(
    "day_of_week"
).agg(
    count("*").alias("num_trips")
)

# Busiest Day
df_weekday.orderBy(
    col("num_trips").desc()
).limit(1).show()

# Slowest Day
df_weekday.orderBy(
    col("num_trips").asc()
).limit(1).show()

+-----------+---------+
|day_of_week|num_trips|
+-----------+---------+
|          5|  1357043|
+-----------+---------+

+-----------+---------+
|day_of_week|num_trips|
+-----------+---------+
|          1|   859905|
+-----------+---------+



In [ ]:
# Does trip distance or num passangers affect tip amount
print("Distance vs Tip:",
      df_trips.stat.corr("trip_distance", "tip_amount"))

print("Passengers vs Tip:",
      df_trips.stat.corr("passenger_count", "tip_amount"))
# Trip distance has a moderate positive correlation with tip amount (0.527),
# while passenger count has almost no correlation with tip amount (0.004)

Distance vs Tip: 0.5269200663652668
Passengers vs Tip: 0.004431051585116288


In [ ]:
# What was the highest "extra" charge and which trip
df_trips.orderBy(col("extra").desc()).limit(1).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-------+
|       1| 2019-01-23 08:58:09|  2019-01-23 08:58:09|            1.0|          0.0|       1.0|                 Y|          24|         264|           2|  355676

### **Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?**
Several potential outliers were identified in the dataset. The fare_amount has a mean of approximately \$12.53 but a maximum of \$623,259.86, which is extremely high compared with the typical fare. There are also negative values for fare_amount, tip_amount, and total_amount, which may represent refunds.

The dataset also contains 55,089 trips with a trip_distance of 0 miles. Some of these records have unusually high fares despite having zero distance. For example, one trip has 9 passengers, 0 miles of distance, and a \$92 fare. These records could represent unusual trips, cancelled trips, or potential data-quality issues.

Therefore, these observations should be investigated further rather than automatically removed, since an outlier does not necessarily mean that the data is incorrect.

In [ ]:
# Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?
df_trips.select(
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "tip_amount",
    "total_amount"
).describe().show()


df_trips.orderBy(
    col("passenger_count").desc()
).select(
    "trip_id",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount"
).limit(10).show()


+-------+------------------+------------------+-----------------+------------------+-----------------+
|summary|     trip_distance|   passenger_count|      fare_amount|        tip_amount|     total_amount|
+-------+------------------+------------------+-----------------+------------------+-----------------+
|  count|           7696617|           7667945|          7696617|           7696617|          7696617|
|   mean|2.8301461681153532|1.5670317144945614|12.52967677747685|1.8208300763883147|15.81065134371489|
| stddev| 3.774548394256295|1.2244198591042095|261.5897471783846|2.4994631914320986|261.8117056584905|
|    min|               0.0|               0.0|           -362.0|             -63.5|           -362.8|
|    max|             831.8|               9.0|        623259.86|            787.25|        623261.66|
+-------+------------------+------------------+-----------------+------------------+-----------------+

+-------+---------------+-------------+-----------+------------+
|trip_i

In [ ]:
df_trips.filter(
    col("trip_distance") == 0
).count()

55089

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [ ]:
# Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

response = requests.get(zone_url)

zone_file = "taxi_zone_lookup.csv"

if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

In [ ]:
df_zones = spark.read.csv(
    zone_file,
    header=True,
    inferSchema=True
)

df_zones.show()
df_zones.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [ ]:
df_zones.createOrReplaceTempView("zones")
spark.sql("""
    SELECT *
    FROM zones
    LIMIT 10
""").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+



In [ ]:
# which borough had most pickups? dropoffs?
# Pickups
df_trips_zones_pickup = df_trips.join(
    df_zones,
    df_trips.PULocationID == df_zones.LocationID,
    "left"
)
df_pickup_borough = df_trips_zones_pickup.groupBy(
    "Borough"
).agg(
    count("*").alias("nb_pickups")
)

df_pickup_borough.orderBy(col("nb_pickups").desc()).limit(1).show()

# Dropoffs
df_trips_zones_dropoff = df_trips.join(
    df_zones,
    df_trips.DOLocationID == df_zones.LocationID,
    "left"
)
df_dropoff_borough = df_trips_zones_dropoff.groupBy(
    "Borough"
).agg(
    count("*").alias("nb_dropoffs")
)

df_dropoff_borough.orderBy(col("nb_dropoffs").desc()).limit(1).show()

+---------+----------+
|  Borough|nb_pickups|
+---------+----------+
|Manhattan|   6950965|
+---------+----------+

+---------+----------+
|  Borough|nb_pickups|
+---------+----------+
|Manhattan|   6817355|
+---------+----------+



In [ ]:
# what are the busy/slow times by borough
from pyspark.sql.functions import hour, count, col

df_borough_hour = df_trips.withColumn(
    "hour",
    hour("tpep_pickup_datetime")
).join(
    df_zones,
    df_trips.PULocationID == df_zones.LocationID,
    "left"
)

df_borough_hour = df_borough_hour.groupBy(
    "Borough",
    "hour"
).agg(
    count("*").alias("num_trips")
)

# Busiest
df_borough_hour.orderBy(
    col("num_trips").desc()
).limit(1).show()

# Slowest
df_borough_hour.orderBy(
    col("num_trips").asc()
).limit(1).show()


+---------+----+---------+
|  Borough|hour|num_trips|
+---------+----+---------+
|Manhattan|  18|   471539|
+---------+----+---------+

+-------+----+---------+
|Borough|hour|num_trips|
+-------+----+---------+
|    EWR|  23|        1|
+-------+----+---------+



In [68]:
# what are the busiest days of the week by borough?
spark.sql("""
    SELECT Borough, weekday, num_trips
    FROM (
        SELECT
            z.Borough,
            dayofweek(t.tpep_pickup_datetime) AS weekday,
            COUNT(*) AS num_trips,
            ROW_NUMBER() OVER (
                PARTITION BY z.Borough
                ORDER BY COUNT(*) DESC
            ) AS rank
        FROM trips t
        LEFT JOIN zones z
            ON t.PULocationID = z.LocationID
        GROUP BY
            z.Borough,
            dayofweek(t.tpep_pickup_datetime)
    )
    WHERE rank = 1
""").show()

+-------------+-------+---------+
|      Borough|weekday|num_trips|
+-------------+-------+---------+
|        Bronx|      5|     3121|
|     Brooklyn|      3|    15779|
|          EWR|      4|       83|
|    Manhattan|      5|  1229554|
|          N/A|      3|      703|
|       Queens|      5|    78972|
|Staten Island|      6|       64|
|      Unknown|      5|    28929|
+-------------+-------+---------+



In [73]:
# what is the average trip distance by borough?
df_trips_zones_pickup.groupBy(
    "Borough"
).agg(
    avg("trip_distance").alias("avg_distance")
).show()


+-------------+------------------+
|      Borough|      avg_distance|
+-------------+------------------+
|       Queens|11.283218499361993|
|          EWR| 2.641098654708519|
|      Unknown| 2.415464130400774|
|     Brooklyn| 4.787677275447492|
|Staten Island|12.503601108033246|
|          N/A| 3.193850899742941|
|    Manhattan|2.2286693358402596|
|        Bronx| 7.233194552098303|
+-------------+------------------+



In [75]:
# what is the average trip fare by borough?
df_trips_zones_pickup.groupBy(
    "Borough"
).agg(
    avg("fare_amount").alias("avg_fare")
).show()
#test

+-------------+------------------+
|      Borough|          avg_fare|
+-------------+------------------+
|       Queens| 35.14462651722029|
|          EWR| 76.24024663677126|
|      Unknown|14.944423051653523|
|     Brooklyn|18.649132800172286|
|Staten Island|45.289861495844896|
|          N/A|  59.5731593830335|
|    Manhattan|10.792468572351568|
|        Bronx| 26.26890543682963|
+-------------+------------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing